In [1]:
# import
import os, sys
import numpy as np
import pandas as pd
import scipy as sp
import scipy.io
from jedi.inference.imports import load_module_from_path
from scipy import stats
from scipy.spatial import distance
from sklearn.cluster import KMeans
from tqdm import tqdm

# import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import datasets
from nilearn import plotting


from nctpy.energies import integrate_u, get_control_inputs
from nctpy.pipelines import ComputeControlEnergy, ComputeOptimizedControlEnergy
from nctpy.metrics import ave_control
from nctpy.utils import matrix_normalization, convert_states_str2int, \
    normalize_state, normalize_weights, get_null_p, get_fdr_p
from nctpy.plotting import roi_to_vtx, null_plot, surface_plot, add_module_lines, set_plotting_params
from null_models.geomsurr import geomsurr
set_plotting_params()

from control_energy_utils import ce_baseline
from control_energy_utils import ce_baseline_all
from control_energy_utils import ce_between
from control_energy_utils import distance_between_states

ModuleNotFoundError: No module named 'nctpy'

CE from baseline to activations

All keywords ce growth chart

In [2]:
# Load keywords
loaded = np.load('keywords_refined.npz', allow_pickle=True)
keywords = loaded['all_keywords'].tolist()

In [7]:
# Initialize dictionary to store results by subject
subject_data = {}

# Define datasets
datasets = ['dHCP', 'BCP', 'HBN', 'HCPD']

# Iterate through all keywords
for keyword in tqdm(keywords, desc="Processing keywords"):
    print(f"\nProcessing keyword: {keyword}")

    try:
        # Run the function for current keyword
        results = ce_baseline_all(
            activation_keyword=keyword,
            datasets=datasets
        )

        # Extract individual dataset results
        dataset_results = results['dataset_results']

        # Process each dataset's results
        for dataset_result in dataset_results:
            dataset_name = dataset_result['dataset']
            ages = dataset_result['age']
            node_energies = dataset_result['node_energy_mean']

            # Create entry for each subject
            for idx, (age, energy) in enumerate(zip(ages, node_energies)):
                # Create unique subject identifier
                subject_id = f"{dataset_name}_{idx}"

                # Initialize subject entry if not exists
                if subject_id not in subject_data:
                    subject_data[subject_id] = {
                        'subject_id': subject_id,
                        'age': age,
                        'study': dataset_name
                    }

                # Add keyword-specific energy value
                subject_data[subject_id][f'node_energy_mean_{keyword}'] = energy

    except Exception as e:
        print(f"Error processing keyword '{keyword}': {str(e)}")
        continue

# Convert dictionary to DataFrame
df = pd.DataFrame.from_dict(subject_data, orient='index')

# Reset index to make subject_id a column (it's already in the data)
df = df.reset_index(drop=True)

# Reorder columns: subject_id, age, study, then all keyword columns
base_cols = ['subject_id', 'age', 'study']
keyword_cols = [col for col in df.columns if col.startswith('node_energy_mean_')]
df = df[base_cols + keyword_cols]

# Save to CSV
output_file = 'ce_baseline_keywords.csv'
df.to_csv(output_file, index=False)

print(f"\nResults saved to {output_file}")
print(f"Shape: {df.shape}")
print(f"Number of subjects: {len(df)}")
print(f"Number of keywords processed: {len(keyword_cols)}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())

Processing keywords:   0%|          | 0/100 [00:00<?, ?it/s]


Processing keyword: action


Processing keywords:   1%|          | 1/100 [01:24<2:18:36, 84.01s/it]


Processing keyword: control


Processing keywords:   2%|▏         | 2/100 [03:26<2:53:53, 106.47s/it]

...


Processing keyword: visual perception


Processing keywords:  99%|█████████▉| 99/100 [2:14:01<01:15, 75.13s/it]


Processing keyword: motivation


Processing keywords: 100%|██████████| 100/100 [2:15:18<00:00, 81.19s/it]



Results saved to ce_baseline_keywords.csv
Shape: (3926, 103)
Number of subjects: 3926
Number of keywords processed: 100

First few rows:
  subject_id       age study  node_energy_mean_action  \
0     dHCP_0 -0.297921  dHCP               960.147503   
1     dHCP_1 -0.297921  dHCP               955.535092   
2     dHCP_2  0.856813  dHCP               988.493410   
3     dHCP_3  0.330254  dHCP               973.531718   
4     dHCP_4  0.725173  dHCP               947.943646   

   node_energy_mean_control  node_energy_mean_initiation  \
0                197.217974                   440.518240   
1                198.483944                   437.021055   
2                202.335449                   443.024793   
3                200.481147                   436.848445   
4                197.894904                   444.126452   

   node_energy_mean_motor  node_energy_mean_movement  \
0              932.398113                 924.443593   
1              945.989717                 934.

CE between activation: exp. emotion and language

In [ ]:
emo = ['emotion', 'recognition', 'anxiety', 'fear']
lan = ['language', 'reading', 'learning', 'verbal']

keywords = emo + lan

# Dictionary to store all results
results = {}

for k1 in keywords:
    for k2 in keywords:
        pair_key = f"{k1}_to_{k2}"

        # ce_between returns a dictionary, not unpacked values
        result = ce_between(k1, k2, 'dHCP')

        # Store the entire result dictionary
        results[pair_key] = result

        print(f"Completed: {k1} → {k2}")

# Create matrix for heatmap
heatmap_data = np.zeros((len(keywords), len(keywords)))

for i, k1 in enumerate(keywords):
    for j, k2 in enumerate(keywords):
        pair_key = f"{k1}_to_{k2}"
        # Calculate average of node_energy_mean across all subjects
        heatmap_data[i, j] = np.mean(results[pair_key]['node_energy_mean'])

# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_data,
            annot=True,  # Show values in cells
            fmt='.3f',   # Format to 3 decimal places
            cmap='coolwarm',  # Color scheme
            xticklabels=keywords,
            yticklabels=keywords,
            cbar_kws={'label': 'Mean Node Energy'})

plt.xlabel('Target Keywords', fontsize=12)
plt.ylabel('Source Keywords', fontsize=12)
plt.title('Control Energy: All Keywords → All Keywords Transitions', fontsize=14)
plt.tight_layout()
plt.show()